# Sustentación en vivo: modificar el ECLC y las pruebas estadísticas

Cuaderno de apoyo para explicar o cambiar en vivo cualquier parte del modelo original y de la validación. Carga el ECLC ya entrenado (el mismo del dashboard) y una muestra de la prueba reservada; cada modificación se evalúa en segundos porque **las probabilidades de los 7 modelos base se calculan una sola vez** y solo se recalcula la regla de combinación.

## 0. Preparación

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

RAIZ = Path.cwd()
sys.path[:0] = [str(RAIZ / "dashboard"), str(RAIZ / "docs"), str(RAIZ / "scripts")]
import src.original_model as om                   # el modelo original (clase EnsambleCompetenciaLocal)
from src import modelos as MOD                    # carga de artefactos del dashboard
from _utils import load_covertype, split_covertype

_, X, y = load_covertype()
X_tr, X_te, y_tr, y_te = split_covertype(X, y)            # 80/20 estratificado, semilla 42
X_m, _, y_m, _ = train_test_split(X_te, y_te, train_size=10_000, stratify=y_te, random_state=42)
y_m = y_m.to_numpy()

eclc = joblib.load(RAIZ / "dashboard/artifacts/ECLC.joblib")
BASES = MOD.metadata("ECLC")["modelos_base"]
t = time.time()
P = np.stack([MOD._proba_base(m, X_m) for m in BASES])   # (M, n, C): probabilidades de los 7 modelos base
print(f"Probabilidades base listas en {time.time() - t:.1f} s; forma {P.shape}")
print(f"Hiperparámetros elegidos por TPE: k = {eclc.k}, beta = {eclc.beta:.2f}, tau = {eclc.tau:.1f}")

def evaluar(nombre, modelo=eclc):
    proba = modelo.combinar(P, X=X_m)
    f1 = f1_score(y_m, np.argmax(proba, 1) + 1, average="macro")
    print(f"{nombre:<45s} F1 macro = {f1:.4f}")
    return f1

Probabilidades base listas en 4.0 s; forma (7, 10000, 7)
Hiperparámetros elegidos por TPE: k = 6, beta = 21.77, tau = 101.3


## 1. Punto de partida: el ECLC frente a cada modelo base (misma muestra)

In [2]:
for m, Pm in zip(BASES, P):
    print(f"{m:<45s} F1 macro = {f1_score(y_m, np.argmax(Pm, 1) + 1, average='macro'):.4f}")
F1_ECLC = evaluar("ECLC (original)")

KNN                                           F1 macro = 0.7736
NB                                            F1 macro = 0.2311
LR                                            F1 macro = 0.5635
DT                                            F1 macro = 0.7536
RF                                            F1 macro = 0.8020
XGB                                           F1 macro = 0.7665
SVM                                           F1 macro = 0.6380


ECLC (original)                               F1 macro = 0.8109


## 2. Mirar por dentro: pesos de una celda

`pesos()` devuelve w[m, c] para cada celda: cuánto confía el ECLC en el modelo m para la clase c. Cada columna suma 1 (regla de Gibbs).

In [3]:
W = eclc.pesos(X=X_m.iloc[[0]])[0]                  # (M, C)
print("Cobertura real:", y_m[0])
display(pd.DataFrame(W, index=BASES, columns=[f"clase {c}" for c in range(1, 8)]).round(3))
print("Cada columna suma:", W.sum(axis=0).round(6))

Cobertura real: 2


,clase 1,clase 2,clase 3,clase 4,clase 5,clase 6,clase 7
KNN,0.023,0.000,0.016,0.008,0.000,0.016,0.042
NB,0.000,0.000,0.000,0.000,0.000,0.000,0.000
LR,0.050,0.007,0.051,0.175,0.003,0.054,0.060
DT,0.000,0.000,0.000,0.000,0.000,0.000,0.000
RF,0.598,0.900,0.554,0.479,0.914,0.650,0.585
XGB,0.293,0.090,0.374,0.297,0.083,0.278,0.306
SVM,0.037,0.002,0.005,0.041,0.000,0.002,0.007


Cada columna suma: [1. 1. 1. 1. 1. 1. 1.]


## 3. Cambiar β (selectividad)

β = 0 → todos los modelos pesan igual (promedio simple). β muy grande → casi solo el mejor modelo local.

In [4]:
for b in [0.0, 1.0, eclc.beta, 200.0]:
    eclc.beta = b
    evaluar(f"beta = {b:.2f}")
eclc.beta = 21.76624112345367                      # se restaura el valor optimizado

beta = 0.00                                   F1 macro = 0.8016


beta = 1.00                                   F1 macro = 0.8019


beta = 21.77                                  F1 macro = 0.8109


beta = 200.00                                 F1 macro = 0.8040


## 4. Cambiar τ (contracción)

τ = 0 → solo la vecindad (pesos inestables con pocos vecinos). τ = ∞ → pesos globales por clase, iguales para todas las celdas.

In [5]:
tau0 = eclc.tau
for t_ in [0.0, tau0, 1e4, np.inf]:
    eclc.tau = t_
    evaluar(f"tau = {t_}")
eclc.tau = tau0

tau = 0.0                                     F1 macro = 0.7737


tau = 101.29197956845728                      F1 macro = 0.8109


tau = 10000.0                                 F1 macro = 0.8031
tau = inf                                     F1 macro = 0.8031


## 5. Cambiar k (tamaño del vecindario)

In [6]:
k0 = eclc.k
for k in [1, 6, 25, 100]:
    eclc.k = k
    evaluar(f"k = {k}")
eclc.k = k0

k = 1                                         F1 macro = 0.8034


k = 6                                         F1 macro = 0.8109


k = 25                                        F1 macro = 0.8023


k = 100                                       F1 macro = 0.8055


## 6. Quitar el núcleo de distancia o el balanceo de clases de los vecinos

In [7]:
eclc.ponderar_distancia = False
evaluar("sin núcleo gaussiano (vecinos equiponderados)")
eclc.ponderar_distancia = True

s0 = eclc.peso_clase_.copy()
print("Pesos por clase de los vecinos (n / (C n_c)):", s0.round(2))
eclc.peso_clase_ = np.ones_like(s0)
evaluar("sin balanceo de clases en los vecinos")
eclc.peso_clase_ = s0

sin núcleo gaussiano (vecinos equiponderados) F1 macro = 0.8089
Pesos por clase de los vecinos (n / (C n_c)): [ 0.39  0.29  2.32 30.23  8.74  4.78  4.05]


sin balanceo de clases en los vecinos         F1 macro = 0.8067


## 7. Modificar el código: selección dura en lugar de ponderación suave

Se reemplaza la función `pesos_gibbs` del módulo por una que da peso 1 al modelo con menor pérdida (equivale a β → ∞, al estilo OLA).

In [8]:
gibbs_original = om.pesos_gibbs

def seleccion_dura(L, beta, eje):
    W = np.zeros_like(L)
    np.put_along_axis(W, np.argmin(L, axis=eje)[:, None], 1.0, axis=eje)
    return W

om.pesos_gibbs = seleccion_dura
evaluar("selección dura (solo el mejor modelo local)")
om.pesos_gibbs = gibbs_original
evaluar("ECLC restaurado")

selección dura (solo el mejor modelo local)   F1 macro = 0.8038


ECLC restaurado                               F1 macro = 0.8109


0.8108808671547898

## 8. Modificar el código: distancia Manhattan en lugar de euclidiana

In [9]:
vecinos_original = om.vecinos_mas_cercanos

def vecinos_manhattan(Z_ref, Z_q, k, tam_bloque=256):
    idx = np.empty((len(Z_q), k), dtype=np.int64); dist = np.empty((len(Z_q), k))
    for i in range(0, len(Z_q), tam_bloque):
        d = np.abs(Z_q[i:i + tam_bloque, None, :] - Z_ref[None, :, :]).sum(axis=2)   # |a - b|_1
        p = np.argpartition(d, k - 1, axis=1)[:, :k]
        dp = np.take_along_axis(d, p, 1); o = np.argsort(dp, 1)
        idx[i:i + tam_bloque] = np.take_along_axis(p, o, 1); dist[i:i + tam_bloque] = np.take_along_axis(dp, o, 1)
    return idx, dist

om.vecinos_mas_cercanos = vecinos_manhattan
t = time.time(); evaluar("distancia Manhattan"); print(f"({time.time() - t:.0f} s)")
om.vecinos_mas_cercanos = vecinos_original

distancia Manhattan                           F1 macro = 0.8110
(70 s)


## 9. Pruebas estadísticas en vivo (30 pliegues de la submuestra)

Se cargan los F1 de los 30 pliegues pareados y se repiten las pruebas con las funciones del proyecto (`scripts/e3_estadistica.py`).

In [10]:
import e3_estadistica as EST
from scipy import stats
from statsmodels.stats.multitest import multipletests

df = EST.cargar()
piv = df.pivot_table(index=["rep", "pliegue"], columns="metodo", values="f1_macro")
print(piv.mean().sort_values(ascending=False).round(4).to_string())

universo = ["ECLC"] + EST.COMPETIDORES
chi2, p = stats.friedmanchisquare(*[piv[c] for c in universo])
print(f"\nFriedman: chi2 = {chi2:.1f}, p = {p:.2e}")

metodo
stacking lineal               0.8088
HGB (estado del arte)         0.8074
ECLC                          0.7908
sin nucleo de distancia       0.7900
sin vecindario (tau = inf)    0.7896
pesos por error global        0.7878
sin pesos por clase           0.7864
base:RF                       0.7857
OLA                           0.7826
sin balanceo de vecinos       0.7799
promedio simple (beta = 0)    0.7767
KNORA-E                       0.7690
sin contraccion (tau = 0)     0.7661
base:KNN                      0.7455
base:XGB                      0.7444
base:DT                       0.7184
base:SVM                      0.6541
base:LR                       0.5638
base:NB                       0.2250

Friedman: chi2 = 236.9, p = 1.01e-46


In [11]:
dif = (piv["ECLC"] - piv["base:RF"]).to_numpy()          # ECLC menos el mejor benchmark, pliegue a pliegue
print(f"Diferencia media {dif.mean():.4f}; ECLC gana en {(dif > 0).sum()} de {len(dif)} pliegues")
print("Wilcoxon unilateral (ECLC mejor):", stats.wilcoxon(dif, alternative="greater", zero_method="zsplit").pvalue)
print("Wilcoxon bilateral              :", stats.wilcoxon(dif, alternative="two-sided", zero_method="zsplit").pvalue)
n_tr, n_te = 26_667, 13_333                                   # tamaños de cada pliegue (3 pliegues de 40.000)
print("t de Nadeau-Bengio (t, p)      :", EST.nadeau_bengio(dif, n_tr, n_te))
print("t pareada SIN corrección       :", stats.ttest_rel(piv["ECLC"], piv["base:RF"], alternative="greater").pvalue)
print("delta de Cliff                 :", EST.cliff(piv["ECLC"], piv["base:RF"]))
print("Bayesiana (P ECLC mejor, P ROPE, P RF mejor):", EST.bayes_signed_rank(dif))
print("Bayesiana con ROPE = 0,002     :", EST.bayes_signed_rank(dif, rope=0.002))

Diferencia media 0.0051; ECLC gana en 27 de 30 pliegues
Wilcoxon unilateral (ECLC mejor): 6.51925802230835e-08
Wilcoxon bilateral              : 1.30385160446167e-07
t de Nadeau-Bengio (t, p)      : (1.5849986377567746, 0.06190615088447159)
t pareada SIN corrección       : 3.14831668149016e-07
delta de Cliff                 : 0.38222222222222224
Bayesiana (P ECLC mejor, P ROPE, P RF mejor): [0.0, 1.0, 0.0]
Bayesiana con ROPE = 0,002     : [0.99985, 0.00015, 0.0]


In [12]:
p_w = [stats.wilcoxon(piv["ECLC"] - piv[c], alternative="greater", zero_method="zsplit").pvalue for c in EST.COMPETIDORES]
tabla = pd.DataFrame({"competidor": EST.COMPETIDORES, "p_sin_ajuste": p_w,
                      "p_Holm": multipletests(p_w, method="holm")[1],
                      "p_Bonferroni": multipletests(p_w, method="bonferroni")[1]})
tabla.style.format({"p_sin_ajuste": "{:.2e}", "p_Holm": "{:.2e}", "p_Bonferroni": "{:.2e}"})

,competidor,p_sin_ajuste,p_Holm,p_Bonferroni
0,base:KNN,9.31e-10,7.45e-09,7.45e-09
1,base:NB,9.31e-10,7.45e-09,7.45e-09
2,base:LR,9.31e-10,7.45e-09,7.45e-09
3,base:DT,9.31e-10,7.45e-09,7.45e-09
4,base:RF,6.52e-08,1.30e-07,5.22e-07
5,base:XGB,9.31e-10,7.45e-09,7.45e-09
6,base:SVM,9.31e-10,7.45e-09,7.45e-09
7,HGB (estado del arte),1.00e+00,1.00e+00,1.00e+00
